# Deepfake Audio Detection with Signal Processing and Machine Learning

**Davi Dai, Yan Zhang** · Course project, PSYCH 186B

Generative AI can now clone a voice from a few seconds of audio, which opens the door to fraud, impersonation and misinformation.
This notebook asks two questions:

1. **Can simple, interpretable acoustic features tell real speech from AI-generated speech?** We compare XGBoost, Random Forest and an LSTM on 26 Librosa features.
2. **Does that detector still work on audio from somewhere else?** We test the trained models on two outside datasets (Fake-or-Real and VCTK).

**Short answer:** in-distribution, yes (about 99% accuracy). Out of distribution, no. Accuracy falls toward chance, so the models learn characteristics of one dataset rather than of "deepfakeness" in general.

| Section | What it does |
|---|---|
| 1 | Setup and data loading |
| 2 | Exploratory data analysis |
| 3 | Model comparison (XGBoost, Random Forest, LSTM) |
| 4 | Stability across seeds and a leakage check |
| 5 | Interpretability: feature selection, importance, SHAP |
| 6 | Feature ablation |
| 7 | Error analysis and a label-shuffle sanity check |
| 8 | Cross-dataset generalization |
| 9 | Conclusions and limitations |

> **Runtime:** about 25 to 40 minutes on a Colab CPU, most of it LSTM training and audio feature extraction. Set `RUN_LSTM = False` below to skip the LSTM.

## 1. Setup

In [ ]:
# In Colab, install the packages that are not preinstalled:
# !pip install -q xgboost shap librosa kagglehub

In [ ]:
import os, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay)
from xgboost import XGBClassifier

SEED = 42
RUN_LSTM = True          # False skips the slow LSTM experiments
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
PALETTE = {"XGBoost": "#C2185B", "Random Forest": "#6A1B9A", "LSTM": "#F9A825"}

if RUN_LSTM:
    import tensorflow as tf
    from tensorflow.keras import Sequential, layers, optimizers
    tf.random.set_seed(SEED)

### Load the DEEP-VOICE dataset

[DEEP-VOICE](https://www.kaggle.com/datasets/birdy654/deep-voice-deepfake-voice-recognition) (Bird & Lotfi, 2023) contains real speech and voice-converted deepfakes of the same speakers.
Each row is a **1-second audio segment** described by 26 acoustic features. Labels are encoded explicitly as `FAKE = 0` and `REAL = 1`.

In [ ]:
import kagglehub

DATA_DIR = kagglehub.dataset_download("birdy654/deep-voice-deepfake-voice-recognition")
csv_path = next(os.path.join(d, f) for d, _, files in os.walk(DATA_DIR)
                for f in files if f.endswith(".csv"))
print("Loading", csv_path)

df = pd.read_csv(csv_path)
LABEL_MAP = {"FAKE": 0, "REAL": 1}
df["LABEL"] = df["LABEL"].map(LABEL_MAP)
FEATURES = [c for c in df.columns if c != "LABEL"]

print(df.shape)
print(df["LABEL"].value_counts().rename({0: "FAKE", 1: "REAL"}))
df.head()

## 2. Exploratory data analysis

In [ ]:
means = df.groupby("LABEL")[FEATURES].mean()
stds = df.groupby("LABEL")[FEATURES].std()
x = np.arange(len(FEATURES)); w = 0.4

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
for ax, stat, name in [(axes[0], means, "Mean"), (axes[1], stds, "Standard deviation")]:
    ax.bar(x - w/2, stat.loc[1], w, label="Real", color="#F9A825")
    ax.bar(x + w/2, stat.loc[0], w, label="Fake", color="#6D4C41")
    ax.set_yscale("symlog"); ax.set_ylabel(f"{name} (symlog)")
    ax.set_title(f"{name} of each feature, real vs. fake"); ax.legend()
axes[1].set_xticks(x); axes[1].set_xticklabels(FEATURES, rotation=90)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(7, 4, figsize=(16, 24))
for ax, col in zip(axes.flat, FEATURES):
    sns.kdeplot(data=df, x=col, hue="LABEL", ax=ax, common_norm=False, fill=True,
                palette={0: "#1E88E5", 1: "#F9A825"}, legend=False)
    ax.set_title(col); ax.set_xlabel("")
for ax in axes.flat[len(FEATURES):]:
    ax.axis("off")
fig.legend(["Real", "Fake"], loc="upper center", ncol=2)
plt.tight_layout(rect=(0, 0, 1, 0.98)); plt.show()

Features whose distributions separate cleanly (for example several MFCCs) are good candidates for discriminating real from fake. Heavily overlapping features add little on their own.

In [ ]:
corr = df.corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Feature correlation matrix (including LABEL)"); plt.show()

print("Features most correlated with the label:")
corr["LABEL"].drop("LABEL").sort_values(key=abs, ascending=False).head(8).round(3)

Neighbouring MFCCs and the spectral shape features (centroid, bandwidth, rolloff) are strongly inter-correlated, so the feature set carries a lot of redundant information. This matters for the ablation results in Section 6.

## 3. Model comparison

**Evaluation protocol.** Stratified 60/20/20 split into train, validation and test sets, with a fixed seed.
The validation set is used only for early stopping and learning curves. The **test set is touched once**, for the final numbers.
ROC-AUC is computed from predicted probabilities. Precision and recall treat **REAL** as the positive class.

In [ ]:
X = df[FEATURES].values
y = df["LABEL"].values

X_trval, X_test, y_trval, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_trval, y_trval, test_size=0.25, stratify=y_trval, random_state=SEED)
print(f"train {len(y_train)} | val {len(y_val)} | test {len(y_test)}")


def evaluate(y_true, y_prob, threshold=0.5):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    out = {"Accuracy": accuracy_score(y_true, y_pred),
           "Precision": precision_score(y_true, y_pred, zero_division=0),
           "Recall": recall_score(y_true, y_pred, zero_division=0),
           "F1": f1_score(y_true, y_pred, zero_division=0),
           "MCC": matthews_corrcoef(y_true, y_pred)}
    out["ROC-AUC"] = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    return out

### Model definitions

* **XGBoost:** gradient-boosted trees with early stopping on the validation set.
* **Random Forest:** 100 trees, a standard tabular baseline.
* **LSTM:** the 26 features are standardised, then turned into a sequence of **17 overlapping windows of 10 features** (step 1: features 1–10, step 2: features 2–11, and so on). This lets a recurrent model scan across the feature vector. The same windowing is reused for every feature set in the ablation, so only the removed feature changes.

In [ ]:
def train_xgb(X_tr, y_tr, X_va, y_va, seed=SEED):
    model = XGBClassifier(n_estimators=1000, learning_rate=0.1, early_stopping_rounds=10,
                          eval_metric="logloss", random_state=seed, n_jobs=-1)
    t0 = time.time()
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return model, time.time() - t0


def train_rf(X_tr, y_tr, seed=SEED):
    model = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    t0 = time.time()
    model.fit(X_tr, y_tr)
    return model, time.time() - t0


N_STEPS = 17

def make_windows(X, n_steps=N_STEPS):
    # (n_samples, n_features) -> (n_samples, n_steps, n_features - n_steps + 1)
    width = X.shape[1] - n_steps + 1
    return np.stack([X[:, i:i + width] for i in range(n_steps)], axis=1)


def build_lstm(input_shape):
    model = Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=True),
        layers.LSTM(64),
        layers.Dense(64, activation="relu"), layers.Dropout(0.3),
        layers.Dense(32, activation="relu"), layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=optimizers.Adam(1e-4), loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model


def train_lstm(X_tr, y_tr, X_va, y_va, epochs=20):
    scaler = StandardScaler().fit(X_tr)          # fit on training data only
    Xtr, Xva = make_windows(scaler.transform(X_tr)), make_windows(scaler.transform(X_va))
    model = build_lstm(Xtr.shape[1:])
    t0 = time.time()
    hist = model.fit(Xtr, y_tr, validation_data=(Xva, y_va),
                     epochs=epochs, batch_size=32, verbose=0)
    return model, scaler, hist, time.time() - t0


def predict_lstm(model, scaler, X):
    return model.predict(make_windows(scaler.transform(X)), verbose=0).ravel()

In [ ]:
results, train_times = {}, {}

xgb, train_times["XGBoost"] = train_xgb(X_train, y_train, X_val, y_val)
results["XGBoost"] = evaluate(y_test, xgb.predict_proba(X_test)[:, 1])

rf, train_times["Random Forest"] = train_rf(X_train, y_train)
results["Random Forest"] = evaluate(y_test, rf.predict_proba(X_test)[:, 1])

if RUN_LSTM:
    lstm, lstm_scaler, lstm_hist, train_times["LSTM"] = train_lstm(X_train, y_train, X_val, y_val)
    results["LSTM"] = evaluate(y_test, predict_lstm(lstm, lstm_scaler, X_test))

results_df = pd.DataFrame(results).T
results_df["Train time (s)"] = pd.Series(train_times)
results_df.round(4)

In [ ]:
if RUN_LSTM:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, key in zip(axes, ["loss", "accuracy"]):
        ax.plot(lstm_hist.history[key], label="train")
        ax.plot(lstm_hist.history[f"val_{key}"], label="validation")
        ax.set_title(f"LSTM {key}"); ax.set_xlabel("Epoch"); ax.legend()
    plt.tight_layout(); plt.show()

In [ ]:
metric_cols = ["Accuracy", "Precision", "Recall", "F1", "MCC", "ROC-AUC"]
ax = results_df[metric_cols].T.plot(kind="bar", figsize=(11, 5), width=0.75,
                                    color=[PALETTE[m] for m in results_df.index])
ax.set_ylim(max(0, results_df[metric_cols].values.min() - 0.03), 1.0)
ax.set_ylabel("Score"); ax.set_title("Test-set performance (single split)")
plt.xticks(rotation=0); plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

ax = pd.Series(train_times).plot(kind="bar", figsize=(6, 4), logy=True,
                                 color=[PALETTE[m] for m in train_times])
for i, v in enumerate(train_times.values()):
    ax.text(i, v * 1.15, f"{v:.1f}s", ha="center")
ax.set_ylabel("Training time (s, log scale)"); ax.set_title("Training time")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

XGBoost matches or beats the LSTM while training two to three orders of magnitude faster, so it gives the best accuracy–efficiency trade-off for this tabular feature set.

## 4. How stable and how trustworthy are these numbers?

### 4a. Variation across random splits
A single split can be lucky. Here the two tree models are re-run on **5 different random splits** to get real error bars. The LSTM is skipped to save time.

In [ ]:
rows = []
for s in range(5):
    Xtv, Xte, ytv, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=s)
    Xtr, Xva, ytr, yva = train_test_split(Xtv, ytv, test_size=0.25, stratify=ytv, random_state=s)
    m, _ = train_xgb(Xtr, ytr, Xva, yva, seed=s)
    rows.append({"Model": "XGBoost", **evaluate(yte, m.predict_proba(Xte)[:, 1])})
    m, _ = train_rf(Xtr, ytr, seed=s)
    rows.append({"Model": "Random Forest", **evaluate(yte, m.predict_proba(Xte)[:, 1])})

seed_df = pd.DataFrame(rows)
summary = seed_df.groupby("Model")[metric_cols].agg(["mean", "std"])
display(summary.round(4))

mean, sd = seed_df.groupby("Model")[metric_cols].mean(), seed_df.groupby("Model")[metric_cols].std()
ax = mean.T.plot(kind="bar", yerr=sd.T, capsize=4, figsize=(10, 4.5), width=0.7,
                 color=[PALETTE[m] for m in mean.index])
ax.set_ylim(max(0, (mean - sd).values.min() - 0.02), 1.0)
ax.set_title("Mean ± SD over 5 random splits"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### 4b. Leakage check: are neighbouring segments from the same recording split apart?
Every row is a 1-second slice of a longer recording. With a random split, slices of the **same clip** can land in both train and test, and a model can score well just by recognising the clip.
The CSV has no clip ID, so as an approximation we group **contiguous blocks of rows** and keep each block entirely in train or entirely in test.
If rows are stored in recording order, this keeps most clips together. If they are not, it behaves like a random split.
A large drop from 4a would point to leakage.

In [ ]:
BLOCK = 50
groups = np.arange(len(y)) // BLOCK
block_rows = []
for s in range(5):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=s)
    tr_idx, te_idx = next(gss.split(X, y, groups))
    gss_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=s)
    tr2, va2 = next(gss_val.split(X[tr_idx], y[tr_idx], groups[tr_idx]))
    Xtr, ytr = X[tr_idx][tr2], y[tr_idx][tr2]
    Xva, yva = X[tr_idx][va2], y[tr_idx][va2]
    m, _ = train_xgb(Xtr, ytr, Xva, yva, seed=s)
    block_rows.append(evaluate(y[te_idx], m.predict_proba(X[te_idx])[:, 1]))

print("XGBoost, random split : acc = {:.4f} ± {:.4f}".format(
    seed_df.query("Model=='XGBoost'")["Accuracy"].mean(), seed_df.query("Model=='XGBoost'")["Accuracy"].std()))
block_df = pd.DataFrame(block_rows)
print("XGBoost, block split  : acc = {:.4f} ± {:.4f}".format(block_df["Accuracy"].mean(), block_df["Accuracy"].std()))

## 5. Which features drive the predictions?
Three complementary views are used: a model-free statistical test (chi²), XGBoost's built-in gain importance, and SHAP values, which show both the size and the direction of each feature's effect.

In [ ]:
scaler_mm = MinMaxScaler().fit(X_train)                # chi2 needs non-negative inputs
kbest = SelectKBest(chi2, k="all").fit(scaler_mm.transform(X_train), y_train)
chi2_rank = pd.Series(kbest.scores_, index=FEATURES).sort_values(ascending=False)
print("Top 5 features by chi²:", list(chi2_rank.index[:5]))

imp = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values()
imp.plot(kind="barh", figsize=(8, 8), color="#C2185B")
plt.title("XGBoost feature importance (gain)"); plt.xlabel("Importance"); plt.tight_layout(); plt.show()

In [ ]:
import shap

X_test_df = pd.DataFrame(X_test, columns=FEATURES)
try:
    shap_values = shap.TreeExplainer(xgb)(X_test_df)
except Exception as e:   # some shap/xgboost version combos clash; XGBoost can compute SHAP values itself
    print("shap.TreeExplainer failed:", e, "-> using XGBoost's built-in SHAP contributions")
    import xgboost
    shap_values = xgb.get_booster().predict(xgboost.DMatrix(X_test_df), pred_contribs=True)[:, :-1]
shap.summary_plot(shap_values, X_test_df, show=True)

In the SHAP plot, each dot is one test segment. Its position shows how far that feature pushed the prediction toward REAL (right) or FAKE (left), and its colour shows whether the feature value was high or low.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

top6 = list(imp.sort_values(ascending=False).index[:6])
fig, ax = plt.subplots(figsize=(14, 7))
PartialDependenceDisplay.from_estimator(xgb, pd.DataFrame(X_train, columns=FEATURES), top6, ax=ax)
fig.suptitle("Partial dependence of P(REAL) on the six most important features")
plt.tight_layout(); plt.show()

## 6. Feature ablation: what happens if the "best" features are removed?
Features are dropped based on three sources: the chi² ranking, the original DEEP-VOICE paper (which reports MFCC2 and MFCC18 as most correlated with the label), and our correlation matrix (RMS, MFCC1, MFCC2).
Every model is retrained on the same train/validation/test split with the remaining features.

In [ ]:
ABLATIONS = {
    "all features":      [],
    "– mfcc2":           ["mfcc2"],
    "– mfcc4":           ["mfcc4"],
    "– mfcc18":          ["mfcc18"],
    "– mfcc2, mfcc18":   ["mfcc2", "mfcc18"],
    "– rms, mfcc1, mfcc2": ["rms", "mfcc1", "mfcc2"],
}

abl_rows = []
for name, drop in ABLATIONS.items():
    keep = [i for i, f in enumerate(FEATURES) if f not in drop]
    Xtr, Xva, Xte = X_train[:, keep], X_val[:, keep], X_test[:, keep]
    row = {"Feature set": name}
    m, _ = train_xgb(Xtr, y_train, Xva, y_val)
    row["XGBoost"] = accuracy_score(y_test, m.predict(Xte))
    m, _ = train_rf(Xtr, y_train)
    row["Random Forest"] = accuracy_score(y_test, m.predict(Xte))
    if RUN_LSTM:
        m, sc, _, _ = train_lstm(Xtr, y_train, Xva, y_val)
        row["LSTM"] = accuracy_score(y_test, (predict_lstm(m, sc, Xte) >= 0.5).astype(int))
    abl_rows.append(row)
    print(row)

ablation_df = pd.DataFrame(abl_rows).set_index("Feature set")
display(ablation_df.round(4))

ax = ablation_df.plot(kind="bar", figsize=(11, 4.5), width=0.75,
                      color=[PALETTE[c] for c in ablation_df.columns])
ax.set_ylim(max(0, ablation_df.values.min() - 0.03), 1.0)
ax.set_ylabel("Test accuracy"); ax.set_title("Accuracy after removing features")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

**Interpretation.** Removing the features that look most discriminative barely changes the tree models.
Because the MFCCs and spectral features are highly correlated (Section 2), the remaining features carry almost the same information, and the trees simply split on them instead.
High importance or high correlation with the label does not mean a feature is *irreplaceable*.

## 7. Error analysis and a sanity check

In [ ]:
y_pred_xgb = xgb.predict(X_test)
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_xgb),
                       display_labels=["FAKE", "REAL"]).plot(cmap="Blues", values_format="d")
plt.title("XGBoost confusion matrix (test set)"); plt.show()

mis = X_test_df.assign(true=y_test, pred=y_pred_xgb).query("true != pred")
print(f"{len(mis)} of {len(y_test)} test segments misclassified "
      f"({(mis.true == 0).sum()} fake→real, {(mis.true == 1).sum()} real→fake)")

# Are the misclassified segments unusual? Compare their feature z-scores to the test set.
z = (mis[FEATURES] - X_test_df.mean()) / X_test_df.std()
z.abs().mean().sort_values(ascending=False).head(8).round(2).rename("mean |z| of misclassified")

**Sanity check (label shuffling).** If the training labels are randomly permuted, any real signal is destroyed, so a sound pipeline should fall to chance (about 50%).
Scoring well above chance here would mean information is leaking through somewhere other than the features.

In [ ]:
rng = np.random.default_rng(SEED)
m_shuf, _ = train_xgb(X_train, rng.permutation(y_train), X_val, rng.permutation(y_val))
pd.Series(evaluate(y_test, m_shuf.predict_proba(X_test)[:, 1])).round(4)

## 8. Does it generalize? Cross-dataset evaluation

The real test of a deepfake detector is audio it has never seen. We extract **the same 26 features, the same way** (1-second segments at 22,050 Hz, averaged over frames) from two external datasets:

* **[Fake-or-Real (FoR)](https://www.kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset):** real speech plus TTS deepfakes from different systems and speakers (balanced sample).
* **[VCTK](https://www.kaggle.com/datasets/colestan/vctk-corpus):** real speech only, from 109 English speakers. Since every clip is real, the only meaningful metric is the share classified as REAL.

The models trained in Section 3 are applied **without any retraining**.

In [ ]:
import librosa

SR, SEG_SECONDS = 22050, 1.0
MAX_FILES = 300            # files per class/dataset; raise for tighter estimates (slower)


def segment_features(sig, sr=SR):
    mfcc = librosa.feature.mfcc(y=sig, sr=sr, n_mfcc=20)
    feats = {
        "chroma_stft": np.mean(librosa.feature.chroma_stft(y=sig, sr=sr)),
        "rms": np.mean(librosa.feature.rms(y=sig)),
        "spectral_centroid": np.mean(librosa.feature.spectral_centroid(y=sig, sr=sr)),
        "spectral_bandwidth": np.mean(librosa.feature.spectral_bandwidth(y=sig, sr=sr)),
        "rolloff": np.mean(librosa.feature.spectral_rolloff(y=sig, sr=sr)),
        "zero_crossing_rate": np.mean(librosa.feature.zero_crossing_rate(y=sig)),
    }
    for i in range(20):
        feats[f"mfcc{i+1}"] = np.mean(mfcc[i])
    return feats


def extract_file(path):
    # Split a clip into 1-second segments (as in DEEP-VOICE) and return one feature row per segment.
    sig, sr = librosa.load(path, sr=SR)
    seg = int(SEG_SECONDS * sr)
    if len(sig) < seg:
        return [segment_features(sig, sr)]
    return [segment_features(sig[s:s + seg], sr) for s in range(0, len(sig) - seg + 1, seg)]


def extract_paths(paths, label):
    rows = []
    for p in paths:
        try:
            for r in extract_file(p):
                rows.append({**r, "LABEL": label, "file": p})   # full path = unique clip ID
        except Exception as e:
            print("skipped", p, "-", e)
    return pd.DataFrame(rows)


def wavs_in(folder, n=MAX_FILES):
    return sorted(os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".wav"))[:n]


def find_dir(root, name, parent=None):
    for d, subdirs, _ in os.walk(root):
        if os.path.basename(d) == name and (parent is None or os.path.basename(os.path.dirname(d)) == parent):
            return d
    raise FileNotFoundError(f"{parent}/{name} not found under {root}")

In [ ]:
FOR_DIR = kagglehub.dataset_download("mohammedabdeldayem/the-fake-or-real-dataset")
for_fake_dir = find_dir(FOR_DIR, "fake", parent="training")
for_real_dir = find_dir(FOR_DIR, "real", parent="training")
print(for_fake_dir, for_real_dir, sep="\n")

t0 = time.time()
for_df = pd.concat([extract_paths(wavs_in(for_fake_dir), 0),
                    extract_paths(wavs_in(for_real_dir), 1)], ignore_index=True)
print(f"FoR: {len(for_df)} segments from {for_df['file'].nunique()} files in {time.time()-t0:.0f}s")
for_df["LABEL"].value_counts().rename({0: "FAKE", 1: "REAL"})

In [ ]:
VCTK_DIR = kagglehub.dataset_download("colestan/vctk-corpus")
wav_root = find_dir(VCTK_DIR, "wav48")
speakers = sorted(d for d in os.listdir(wav_root) if os.path.isdir(os.path.join(wav_root, d)))
per_speaker = max(1, MAX_FILES // 30)
vctk_paths = [p for spk in speakers[:30] for p in wavs_in(os.path.join(wav_root, spk), per_speaker)]

t0 = time.time()
vctk_df = extract_paths(vctk_paths, 1)
print(f"VCTK: {len(vctk_df)} segments from {len(vctk_paths)} files, {min(30, len(speakers))} speakers, {time.time()-t0:.0f}s")

In [ ]:
X_for, y_for = for_df[FEATURES].values, for_df["LABEL"].values
X_vctk = vctk_df[FEATURES].values

models = {"XGBoost": lambda Z: xgb.predict_proba(Z)[:, 1],
          "Random Forest": lambda Z: rf.predict_proba(Z)[:, 1]}
if RUN_LSTM:
    models["LSTM"] = lambda Z: predict_lstm(lstm, lstm_scaler, Z)

cross_rows = []
for name, predict in models.items():
    r_for = evaluate(y_for, predict(X_for))
    p_vctk = predict(X_vctk)
    cross_rows.append({"Model": name,
                       "DEEP-VOICE test acc": results[name]["Accuracy"],
                       "FoR acc": r_for["Accuracy"], "FoR ROC-AUC": r_for["ROC-AUC"],
                       "FoR MCC": r_for["MCC"],
                       "VCTK: % called REAL": (p_vctk >= 0.5).mean()})
cross_df = pd.DataFrame(cross_rows).set_index("Model")
cross_df.round(3)

In [ ]:
plot_cols = ["DEEP-VOICE test acc", "FoR acc", "VCTK: % called REAL"]
ax = cross_df[plot_cols].T.plot(kind="bar", figsize=(10, 4.5), width=0.75,
                                color=[PALETTE[m] for m in cross_df.index])
ax.axhline(0.5, ls="--", c="grey", lw=1); ax.text(2.45, 0.51, "chance", color="grey", ha="right")
ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
ax.set_title("In-distribution vs. cross-dataset performance")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### Why does performance collapse? A look at the distribution shift
If we standardise every dataset's features using the **DEEP-VOICE** mean and SD, features that sit far from 0 in the outside datasets are ones whose values the model never saw during training.

In [ ]:
mu, sd = df[FEATURES].mean(), df[FEATURES].std()
shift = pd.DataFrame({
    "FoR real": ((for_df.query("LABEL==1")[FEATURES].mean() - mu) / sd),
    "FoR fake": ((for_df.query("LABEL==0")[FEATURES].mean() - mu) / sd),
    "VCTK real": ((vctk_df[FEATURES].mean() - mu) / sd),
})
plt.figure(figsize=(6, 9))
sns.heatmap(shift, cmap="coolwarm", center=0, annot=True, fmt=".1f", cbar_kws={"label": "z vs. DEEP-VOICE"})
plt.title("Feature shift relative to DEEP-VOICE"); plt.tight_layout(); plt.show()

### Does seeing a little outside data help?
Finally, we add **half of the FoR files** to the training set and test on the other half. The split is grouped by file, so no clip appears on both sides.
If accuracy recovers, the features *can* separate real from fake in FoR. The original model simply never learned what FoR-style audio looks like.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
tr_idx, te_idx = next(gss.split(X_for, y_for, groups=for_df["file"]))

X_mix = np.vstack([X_train, X_for[tr_idx]]); y_mix = np.concatenate([y_train, y_for[tr_idx]])
xgb_mix, _ = train_xgb(X_mix, y_mix, X_val, y_val)

print("FoR held-out files, DEEP-VOICE-only model :",
      round(accuracy_score(y_for[te_idx], xgb.predict(X_for[te_idx])), 3))
print("FoR held-out files, + half of FoR in train:",
      round(accuracy_score(y_for[te_idx], xgb_mix.predict(X_for[te_idx])), 3))
print("DEEP-VOICE test acc of mixed model         :",
      round(accuracy_score(y_test, xgb_mix.predict(X_test)), 3))

## 9. Conclusions

* **In-distribution, hand-crafted acoustic features are enough.** All three models reach about 99% accuracy on DEEP-VOICE. XGBoost is the most accurate and trains in about a second, while the LSTM takes minutes for no gain.
* **The tree models are robust to feature removal**, because the MFCC and spectral features are highly redundant. Removing the "most important" features shifts the model onto correlated substitutes.
* **Generalization is the real problem.** On outside audio (FoR, VCTK), performance drops toward chance and real speakers get flagged as fake. The models learn the recording conditions, speakers and synthesis method of one dataset, not a general signature of synthetic speech.
* **Adding a small amount of outside data largely recovers performance**, which suggests the bottleneck is training-data diversity rather than the feature set itself.

### Limitations
* DEEP-VOICE is small (8 speakers, one voice-conversion system), and its 1-second segments come from a handful of long recordings. Even with the block split in Section 4b, in-dataset scores are probably optimistic.
* Features are frame-averaged over 1 second, which throws away temporal information that modern detectors rely on.
* External datasets were subsampled (`MAX_FILES`) for runtime. Raise it for tighter estimates.

### Next steps
Train on a mix of synthesis systems (for example ASVspoof 2019/2021 plus FoR), evaluate with speaker-disjoint splits, and compare against learned representations such as raw-waveform CNNs or wav2vec 2.0 embeddings.